# Cross-lingual transfer: English supervision → Turkish news

Fine-tunes an encoder on an English political-bias news dataset with stratified 5-fold cross-validation, evaluates it in English, then applies the fold ensemble zero-shot to the Turkish test set used in the main notebook. This is the cross-lingual experiment reported in the paper.


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
! pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00


In [5]:
import json
import numpy as np
import pandas as pd
import os, random, json
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)


In [6]:
os.environ["WANDB_DISABLED"] = "true"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("cuda_available:", torch.cuda.is_available())
print("torch.version.cuda:", torch.version.cuda)

cuda_available: True
torch.version.cuda: 12.6


In [7]:
import pandas as pd

df = pd.read_json(
    "news_article_dataset.json",
    lines=True,          # 🔥 ÖNEMLİ
    encoding="utf-8"
)

print(df.shape)
print(df.columns)
print(df.head())

(8478, 14)
Index(['Title of Headline Roundup', 'description', 'Topics', 'Date',
       'url_story', 'left_story_title', 'center_story_title',
       'right_story_title', 'right_story_url', 'right_story_text',
       'center_story_url', 'center_story_text', 'left_story_url',
       'left_story_text'],
      dtype='object')
   Title of Headline Roundup description       Topics       Date  \
0           Romney in London        None    Elections 2012-07-26   
1     Romney's Overseas Trip        None    Elections 2012-08-01   
2  Biden Comment Controversy        None    Elections 2012-08-16   
3               Romney Taxes        None    Elections 2012-08-17   
4  Skinnydipping Congressman        None  US Congress 2012-08-20   

                          url_story  \
0              /story/romney-london   
1      /story/romneys-overseas-trip   
2  /story/biden-comment-controversy   
3               /story/romney-taxes   
4  /story/skinnydipping-congressman   

                                

In [8]:
print("Raw df shape:", df.shape)
print(df.columns)

Raw df shape: (8478, 14)
Index(['Title of Headline Roundup', 'description', 'Topics', 'Date',
       'url_story', 'left_story_title', 'center_story_title',
       'right_story_title', 'right_story_url', 'right_story_text',
       'center_story_url', 'center_story_text', 'left_story_url',
       'left_story_text'],
      dtype='object')


In [9]:
print(df.columns)
print(df.head(3))


Index(['Title of Headline Roundup', 'description', 'Topics', 'Date',
       'url_story', 'left_story_title', 'center_story_title',
       'right_story_title', 'right_story_url', 'right_story_text',
       'center_story_url', 'center_story_text', 'left_story_url',
       'left_story_text'],
      dtype='object')
   Title of Headline Roundup description     Topics       Date  \
0           Romney in London        None  Elections 2012-07-26   
1     Romney's Overseas Trip        None  Elections 2012-08-01   
2  Biden Comment Controversy        None  Elections 2012-08-16   

                          url_story  \
0              /story/romney-london   
1      /story/romneys-overseas-trip   
2  /story/biden-comment-controversy   

                                    left_story_title  \
0                      Romney's Olympics false start   
1  Was Romney's trip 'a great success' or gaffe-f...   
2            Mission Impossible: Managing Joe Biden    

                                  center

In [10]:
# ==========================================================
# 2) WIDE -> LONG (same idea: build one text column + label)
# ==========================================================
rows = []
for _, row in df.iterrows():
    if pd.notna(row.get("left_story_text", None)) and str(row["left_story_text"]).strip():
        rows.append({"text": row["left_story_text"], "bias_label": "left"})
    if pd.notna(row.get("center_story_text", None)) and str(row["center_story_text"]).strip():
        rows.append({"text": row["center_story_text"], "bias_label": "center"})
    if pd.notna(row.get("right_story_text", None)) and str(row["right_story_text"]).strip():
        rows.append({"text": row["right_story_text"], "bias_label": "right"})

df_long = pd.DataFrame(rows).dropna().reset_index(drop=True)
print("df_long shape:", df_long.shape)
print(df_long["bias_label"].value_counts())

df_long shape: (24505, 2)
bias_label
left      8430
right     8375
center    7700
Name: count, dtype: int64


In [11]:
# =========================================
# 3) LABEL MAP (same as TR)
# =========================================
label2id = {"left": 0, "center": 1, "right": 2}
id2label = {v: k for k, v in label2id.items()}

df_long = df_long[df_long["bias_label"].isin(label2id)].copy()
df_long["label"] = df_long["bias_label"].map(label2id)

In [12]:
# ===================================================
from sklearn.model_selection import train_test_split

temp_df, test_df = train_test_split(
    df_long[["text", "label"]],
    test_size=0.20,
    random_state=SEED,
    stratify=df_long["label"]
)
print(f"Train+Val (temp_df): {temp_df.shape}, Test (test_df): {test_df.shape}")

Train+Val (temp_df): (19604, 2), Test (test_df): (4901, 2)


In [13]:
! pip install evaluate

In [14]:
# =========================================
# 5) XLM-R CV ENSEMBLE (TR clone)
# =========================================
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

In [15]:
# =========================================
# 5) XLM-R CV ENSEMBLE (TR clone)
# =========================================
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

# ---- CONFIG (same style) ----
MODEL_NAME    = "xlm-roberta-base"
MAX_LENGTH    = 128
BATCH_SIZE    = 16
NUM_EPOCHS    = 20
LEARNING_RATE = 1e-5
N_SPLITS      = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

accuracy = evaluate.load("accuracy")
f1_mac   = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_mac.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

# ---- Fixed hold-out test set (same) ----
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True)) \
    .map(tokenize, batched=True) \
    .remove_columns(["text"]) \
    .with_format("torch")

# ---- CV setup on temp_df (same) ----
X, y = temp_df["text"].values, temp_df["label"].values
skf  = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_results, best_ckpts = [], []

# save per-fold logits like TR code
LOG_DIR = "en_xlmr_logits"
os.makedirs(LOG_DIR, exist_ok=True)

for fold, (tr_idx, vl_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n===== FOLD {fold}/{N_SPLITS} =====")

    tr_ds = Dataset.from_pandas(temp_df.iloc[tr_idx].reset_index(drop=True)) \
        .map(tokenize, batched=True) \
        .remove_columns(["text"]) \
        .with_format("torch")

    vl_ds = Dataset.from_pandas(temp_df.iloc[vl_idx].reset_index(drop=True)) \
        .map(tokenize, batched=True) \
        .remove_columns(["text"]) \
        .with_format("torch")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(id2label),
        id2label={int(k): str(v) for k, v in id2label.items()},
        label2id={str(k): int(v) for k, v in label2id.items()},
        # English base checkpoint is already CLS, so no need for ignore_mismatched_sizes
    )

    args = TrainingArguments(
        output_dir=f"./en_xlmr_cv_fold_{fold}",
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.05,
        seed=SEED,
        report_to=[]
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=vl_ds,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()
    fold_results.append(trainer.evaluate())
    best_ckpts.append(trainer.state.best_model_checkpoint)

    # dump raw test logits for this fold (same)
    pr_test = trainer.predict(test_ds)
    np.save(f"{LOG_DIR}/logits_fold{fold}.npy", pr_test.predictions.astype("float32"))

# ---- CV Summary (same) ----
cv_df = pd.DataFrame(fold_results)
print("\n===== CV MEAN METRICS =====\n", cv_df.mean(numeric_only=True))

# ---- Soft voting ensemble on test set (same) ----
all_logits = []
for fold in range(1, N_SPLITS + 1):
    all_logits.append(np.load(f"{LOG_DIR}/logits_fold{fold}.npy"))

avg_logits = np.mean(np.stack(all_logits, axis=0), axis=0)
y_pred_ens = np.argmax(avg_logits, axis=1)
y_true     = test_ds["label"]

print("\n ENSEMBLE CLASSIFICATION REPORT:\n",
      classification_report(y_true, y_pred_ens,
                            target_names=[id2label[i] for i in range(len(id2label))]))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred_ens))


===== FOLD 1/5 =====


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.096900,1.081847,0.427697,0.427319
2,1.055500,0.999885,0.510839,0.506285
3,0.974600,0.992362,0.528947,0.506921
4,0.892400,0.952019,0.554960,0.554736
5,0.810000,1.009404,0.553685,0.551801
6,0.722400,1.098329,0.535068,0.533482
7,0.629200,1.073007,0.548074,0.545928



===== FOLD 2/5 =====


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.093300,1.057093,0.454221,0.448924
2,1.024700,0.991827,0.507779,0.507295
3,0.938700,0.965948,0.536598,0.533436
4,0.853400,1.035810,0.538128,0.530638
5,0.767000,1.078085,0.541188,0.539689
6,0.675700,1.084785,0.534302,0.533696



===== FOLD 3/5 =====


...


===== FOLD 4/5 =====


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.088200,1.058353,0.455241,0.441175
2,1.016100,0.989082,0.527672,0.519665
3,0.931100,0.949771,0.554195,0.553222
4,0.847800,1.063011,0.527416,0.518599
5,0.767400,1.032828,0.558531,0.557430
6,0.674300,1.197311,0.540678,0.539939



===== FOLD 5/5 =====


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.091800,1.071878,0.442092,0.438410
2,1.025500,1.000682,0.510969,0.503112
3,0.940700,0.978875,0.544133,0.540341
4,0.844600,1.038753,0.548724,0.544887
5,0.766200,1.059040,0.544388,0.544349
6,0.682600,1.142859,0.550255,0.550089



===== CV MEAN METRICS =====
 eval_loss                    0.963138
eval_accuracy                0.544736
eval_f1_macro                0.541880
eval_runtime                 7.403000
eval_samples_per_second    529.724000
eval_steps_per_second       33.208800
epoch                        6.200000
dtype: float64

 ENSEMBLE CLASSIFICATION REPORT:
               precision    recall  f1-score   support

        left       0.58      0.55      0.56      1686
      center       0.55      0.51      0.53      1540
       right       0.57      0.64      0.60      1675

    accuracy                           0.57      4901
   macro avg       0.57      0.57      0.57      4901
weighted avg       0.57      0.57      0.57      4901


Confusion Matrix:
 [[ 920  342  424]
 [ 369  791  380]
 [ 298  303 1074]]


In [16]:
import os, shutil
from google.colab import drive

drive.mount("/content/drive")

# 1) en iyi fold checkpoint (eval_loss en düşük) seç
best_fold_idx = int(np.argmin([r["eval_loss"] for r in fold_results]))  # 0-based
best_path = best_ckpts[best_fold_idx]
print("Best fold checkpoint:", best_path)

# 2) Drive’a export klasörü
EXPORT_DIR = "/content/drive/MyDrive/en_xlmr_ft_best"
if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)

# 3) model + tokenizer save
best_model = AutoModelForSequenceClassification.from_pretrained(best_path)
best_model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

print("Saved to:", EXPORT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Best fold checkpoint: ./en_xlmr_cv_fold_4/checkpoint-2943
Saved to: /content/drive/MyDrive/en_xlmr_ft_best


## Zero-shot evaluation on the Turkish test set


In [17]:
import pandas as pd, hashlib, uuid
from sklearn.model_selection import train_test_split

SEED = 42

RAW_CSV = "all_sources_combined_by_keywords.csv"

df_raw = pd.read_csv(RAW_CSV).iloc[:, :7]

allowed_sources = ["BirGün", "Sözcü", "Habertürk", "Euronews",
                   "Milliyet", "Diriliş Postası"]
df_raw = df_raw[df_raw["Source"].isin(allowed_sources)].reset_index(drop=True)

cols_keep = ["Source", "Title", "URL", "Content", "search_keyword"]
df = df_raw[cols_keep].copy()

# 30 kelime filtresi
df = df[df["Content"].apply(lambda x: isinstance(x, str) and len(x.split()) >= 30)]
df.reset_index(drop=True, inplace=True)

bias_map = {
    "BirGün": "left",
    "Sözcü": "left",
    "Habertürk": "center",
    "Euronews": "center",
    "Milliyet": "right",
    "Diriliş Postası": "right"
}
df["bias_label"] = df["Source"].map(bias_map)

label2id = {"left": 0, "center": 1, "right": 2}
id2label = {v: k for k, v in label2id.items()}

df["label"] = df["bias_label"].map(label2id)


In [18]:
temp_df_tr, test_df_tr = train_test_split(
    df[["Content", "label"]],
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"]
)

print("TR Train+Val:", temp_df_tr.shape)
print("TR Test:", test_df_tr.shape)


TR Train+Val: (2330, 2)
TR Test: (583, 2)


In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix


In [20]:
MODEL_DIR = "/content/drive/MyDrive/en_xlmr_ft_best"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)


In [21]:
def tokenize_tr(batch):
    return tokenizer(
        batch["Content"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tr_test_ds = Dataset.from_pandas(
    test_df_tr.reset_index(drop=True)
).map(
    tokenize_tr, batched=True
).remove_columns(
    ["Content"]
).with_format("torch")


In [22]:
trainer = Trainer(
    model=model,
    tokenizer=tokenizer
)

pred = trainer.predict(tr_test_ds)

logits = pred.predictions
y_true = pred.label_ids
y_pred = np.argmax(logits, axis=1)


In [23]:
print("\n=== English FT model → Turkish TEST set ===\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(3)]
))

print("Confusion Matrix:\n",
      confusion_matrix(y_true, y_pred))



=== English FT model → Turkish TEST set ===

              precision    recall  f1-score   support

        left       0.28      0.03      0.06       223
      center       0.37      0.33      0.35       163
       right       0.36      0.75      0.48       197

    accuracy                           0.36       583
   macro avg       0.34      0.37      0.30       583
weighted avg       0.33      0.36      0.28       583

Confusion Matrix:
 [[  7  48 168]
 [ 12  54  97]
 [  6  44 147]]


In [24]:
np.save("logits_en_ft_on_tr_test.npy", logits.astype("float32"))
np.save("labels_tr_test.npy", y_true.astype("int16"))


In [25]:

# =========================================
# 6) BASELINE (pre-trained, no FT) (same)
# =========================================
print("\n Baseline (pre-trained) evaluation...")

baseline_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

baseline_trainer = Trainer(
    model=baseline_model,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

baseline_preds   = baseline_trainer.predict(test_ds)
baseline_logits  = baseline_preds.predictions
baseline_labels  = baseline_preds.label_ids
baseline_y_pred  = np.argmax(baseline_logits, axis=1)

print("\n BASELINE Classification Report:")
print(classification_report(
    baseline_labels,
    baseline_y_pred,
    target_names=[id2label[i] for i in range(3)]
))

cm_base = confusion_matrix(baseline_labels, baseline_y_pred)
print("\nConfusion Matrix - Baseline:\n", cm_base)

# Optional: save logits like TR
np.save("logits_en_xlmr.npy", avg_logits.astype(np.float32))
print("Saved ensemble logits -> logits_en_xlmr.npy")


 Baseline (pre-trained) evaluation...



 BASELINE Classification Report:
              precision    recall  f1-score   support

        left       0.00      0.00      0.00      1686
      center       0.00      0.00      0.00      1540
       right       0.34      1.00      0.51      1675

    accuracy                           0.34      4901
   macro avg       0.11      0.33      0.17      4901
weighted avg       0.12      0.34      0.17      4901


Confusion Matrix - Baseline:
 [[   0    0 1686]
 [   0    0 1540]
 [   0    0 1675]]
Saved ensemble logits -> logits_en_xlmr.npy
